In [2]:
# -*- coding: utf-8 -*-
"""
Домашнее задание – реализация классов Account и CreditAccount
"""

from datetime import datetime
from typing import List, Dict, Any

In [3]:
class Account:
    """
    Базовый банковский счёт.

    Атрибуты:
        holder (str)          – имя владельца счёта;
        __balance (float)     – текущий баланс (приватный);
        _history (list)       – список словарей, описывающих операции.
    """

    def __init__(self, account_holder: str, balance: float = 0.0):
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        self.holder: str = account_holder
        self.__balance: float = float(balance)
        self._history: List[Dict[str, Any]] = []

        # фиксируем создание счёта как первую запись
        self._add_record(
            op_type="init",
            amount=balance,
            status="success",
            extra={"note": "Account created"}
        )

    # --------------------------------------------------------------------- #
    # Внутренний метод для добавления записи в историю
    def _add_record(
        self,
        op_type: str,
        amount: float,
        status: str,
        extra: Dict[str, Any] = None
    ) -> None:
        record = {
            "type": op_type,
            "amount": amount,
            "datetime": datetime.now(),
            "balance": self.__balance,
            "status": status
        }
        if extra:
            record.update(extra)
        self._history.append(record)

    # --------------------------------------------------------------------- #
    # Публичные методы
    def deposit(self, amount: float) -> bool:
        """Пополнение счёта. Возвращает True при успехе."""
        if amount <= 0:
            self._add_record(
                op_type="deposit",
                amount=amount,
                status="fail",
                extra={"reason": "non‑positive amount"}
            )
            return False

        self.__balance += amount
        self._add_record(
            op_type="deposit",
            amount=amount,
            status="success"
        )
        return True

    def withdraw(self, amount: float) -> bool:
        """Снятие средств. Возвращает True при успехе."""
        if amount <= 0:
            self._add_record(
                op_type="withdraw",
                amount=amount,
                status="fail",
                extra={"reason": "non‑positive amount"}
            )
            return False

        if amount > self.__balance:
            # недостаточно средств – фиксируем неудачу
            self._add_record(
                op_type="withdraw",
                amount=amount,
                status="fail",
                extra={"reason": "insufficient funds"}
            )
            return False

        self.__balance -= amount
        self._add_record(
            op_type="withdraw",
            amount=amount,
            status="success"
        )
        return True

    def get_balance(self) -> float:
        """Текущий баланс."""
        return self.__balance

    def get_history(self) -> List[Dict[str, Any]]:
        """Возвращает копию истории операций."""
        return list(self._history)

In [4]:
class CreditAccount(Account):
    """
    Кредитный счёт. Позволяет иметь отрицательный баланс,
    но не ниже -credit_limit.
    """

    def __init__(
        self,
        account_holder: str,
        balance: float = 0.0,
        credit_limit: float = 0.0
    ):
        if credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным")
        self.credit_limit: float = float(credit_limit)
        super().__init__(account_holder, balance)

        # фиксируем параметр кредитного лимита
        self._add_record(
            op_type="credit_init",
            amount=credit_limit,
            status="success",
            extra={"note": "Credit limit set"}
        )

    # --------------------------------------------------------------------- #
    def available_credit(self) -> float:
        """
        Сколько кредитных средств ещё доступно:
        credit_limit + текущий баланс (может быть отрицательным).
        """
        return self.credit_limit + self.get_balance()

    # --------------------------------------------------------------------- #
    def withdraw(self, amount: float) -> bool:
        """
        Переопределённый метод снятия.
        Позволяет уйти в минус, но не ниже -credit_limit.
        """
        if amount <= 0:
            self._add_record(
                op_type="withdraw",
                amount=amount,
                status="fail",
                extra={"reason": "non‑positive amount"}
            )
            return False

        # проверяем, не превысит ли лимит
        if self.get_balance() - amount < -self.credit_limit:
            self._add_record(
                op_type="withdraw",
                amount=amount,
                status="fail",
                extra={"reason": "credit limit exceeded"}
            )
            return False

        # операция проходит, возможно используется кредит
        used_credit = amount > self.get_balance()
        # обновляем баланс (может стать отрицательным)
        new_balance = self.get_balance() - amount
        # Прямой доступ к приватному атрибуту через name‑mangling
        self._Account__balance = new_balance

        self._add_record(
            op_type="withdraw",
            amount=amount,
            status="success",
            extra={"used_credit": used_credit}
        )
        return True

In [5]:
# Создаём обычный счёт
acc = Account("Иван Иванов", balance=1000.0)

print("Начальный баланс:", acc.get_balance())
acc.deposit(500)
acc.withdraw(200)
acc.withdraw(2000)          # попытка снять больше, чем есть
print("\nТекущий баланс:", acc.get_balance())

print("\nИстория операций (Account):")
for rec in acc.get_history():
    print(rec)

# --------------------------------------------------------------
# Создаём кредитный счёт
cred = CreditAccount("Мария Петрова", balance=300.0, credit_limit=500.0)

print("\n\nКредитный счёт – начальный баланс:", cred.get_balance())
print("Доступный кредит:", cred.available_credit())

cred.withdraw(600)          # использует часть кредита
cred.deposit(200)
cred.withdraw(500)          # попытка превысить лимит (не пройдет)

print("\nТекущий баланс (CreditAccount):", cred.get_balance())
print("Доступный кредит:", cred.available_credit())

print("\nИстория операций (CreditAccount):")
for rec in cred.get_history():
    print(rec)

Начальный баланс: 1000.0

Текущий баланс: 1300.0

История операций (Account):
{'type': 'init', 'amount': 1000.0, 'datetime': datetime.datetime(2025, 12, 15, 14, 51, 32, 762841), 'balance': 1000.0, 'status': 'success', 'note': 'Account created'}
{'type': 'deposit', 'amount': 500, 'datetime': datetime.datetime(2025, 12, 15, 14, 51, 32, 763141), 'balance': 1500.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 200, 'datetime': datetime.datetime(2025, 12, 15, 14, 51, 32, 763188), 'balance': 1300.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 2000, 'datetime': datetime.datetime(2025, 12, 15, 14, 51, 32, 763230), 'balance': 1300.0, 'status': 'fail', 'reason': 'insufficient funds'}


Кредитный счёт – начальный баланс: 300.0
Доступный кредит: 800.0

Текущий баланс (CreditAccount): -100.0
Доступный кредит: 400.0

История операций (CreditAccount):
{'type': 'init', 'amount': 300.0, 'datetime': datetime.datetime(2025, 12, 15, 14, 51, 32, 763551), 'balance': 300.0, 'status': 'success',